# Feature Engineering
Create a Dataframe for model input:
- Feature creation
- Daily aggregation of features
- Vizualization

# Import Libraries and Data

In [1]:
# --- Standard Libraries ---
import os
import re
from collections import Counter

# --- Data Science ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# --- NLP ---
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from langdetect import detect, DetectorFactory, LangDetectException

# --- Transformers ---
import torch
from torch.nn.functional import sigmoid
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BertTokenizer,
    BertForSequenceClassification
)
from scipy.special import softmax

# --- Utils ---
from tqdm.auto import tqdm
from IPython.display import display

# --- Setup ---
tqdm.pandas()
nltk.download("punkt")
nltk.download("wordnet")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\paull\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\paull\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

# Kontrollvariablen

In [3]:
# Uses shorter time periods for testing purposes
TEST = True

# Wenn nur einzelne Spalten/Features hinzugefügt werden sollen, bitte unten den Block 'neue Features zur CSV hinzufügen' entsprechend anpassen. Die bestehende final_daily_df csv wird dann in ein df geladen und die neuen Spalten werden dazugemerged und die csv wieder abgespeichert.
einzelne_features_zur_bestehenden_CSV_hinzufügen = False # default = False

# wenn True, wird die final_daily_df CSV ganz neu zusammengestellt.
vollstaendige_neuerstellung_der_csv = False # default = False

In [ ]:
musk_twitter_data_all = pd.read_csv(os.path.join('cleaned', 'musk_twitter_data_all.csv'),parse_dates=["createdAt"])
musk_twitter_data_nlp = pd.read_csv(os.path.join('cleaned', 'musk_twitter_data_nlp.csv'),parse_dates=["createdAt"])

for df in (musk_twitter_data_all, musk_twitter_data_nlp):
    df["isRetweet"] = df["isRetweet"].astype(str).str.lower()
    df["possiblySensitive"] = df["possiblySensitive"].astype(str).str.lower()
    df["fullText"] = df["fullText"].astype(str)

musk_twitter_data_nlp["text_raw"] = musk_twitter_data_nlp["text_raw"].astype(str)
musk_twitter_data_nlp["text_lemmatized"] = musk_twitter_data_nlp["text_lemmatized"].astype(str)

for df in (musk_twitter_data_all, musk_twitter_data_nlp):
    df["date"] = df["createdAt"].dt.date

if TEST:
    start_date = pd.to_datetime("2025-01-01").date()
else:
    start_date = pd.to_datetime("2015-01-01").date()

end_date = musk_twitter_data_all["date"].max()

mask_all = (musk_twitter_data_all["date"] >= start_date) & (musk_twitter_data_all["date"] <= end_date)
musk_twitter_data_all = musk_twitter_data_all.loc[mask_all].reset_index(drop=True)

mask_nlp = (musk_twitter_data_nlp["date"] >= start_date) & (musk_twitter_data_nlp["date"] <= end_date)
musk_twitter_data_nlp = musk_twitter_data_nlp.loc[mask_nlp].reset_index(drop=True)

final_daily_df_base = pd.DataFrame({
    'date': pd.date_range(start=start_date, end=end_date)
})
final_daily_df_base["date"] = final_daily_df_base["date"].dt.date 

print("NLPTweets:", musk_twitter_data_nlp.shape, "AllTweets:", musk_twitter_data_all.shape)
musk_twitter_data_nlp.info()
musk_twitter_data_all.info()

C:\Users\paull\AppData\Local\Temp\ipykernel_22664\1618114294.py:1: DtypeWarning: Columns (11,16,17,18,19) have mixed types. Specify dtype option on import or set low_memory=False.
  musk_twitter_data_all = pd.read_csv(os.path.join('cleaned', 'musk_twitter_data_all.csv'),parse_dates=["createdAt"])
C:\Users\paull\AppData\Local\Temp\ipykernel_22664\1618114294.py:2: DtypeWarning: Columns (11,16,17,18,19) have mixed types. Specify dtype option on import or set low_memory=False.
  musk_twitter_data_nlp = pd.read_csv(os.path.join('cleaned', 'musk_twitter_data_nlp.csv'),parse_dates=["createdAt"])


NLPTweets: (4122, 28) AllTweets: (6327, 25)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4122 entries, 0 to 4121
Data columns (total 28 columns):
 #   Column                    Non-Null Count  Dtype              
---  ------                    --------------  -----              
 0   id                        4122 non-null   int64              
 1   url                       4122 non-null   object             
 2   twitterUrl                4122 non-null   object             
 3   fullText                  4122 non-null   object             
 4   retweetCount              4122 non-null   float64            
 5   replyCount                4122 non-null   float64            
 6   likeCount                 4122 non-null   float64            
 7   quoteCount                4122 non-null   float64            
 8   viewCount                 4122 non-null   float64            
 9   createdAt                 4122 non-null   datetime64[ns, UTC]
 10  bookmarkCount             4122 non-null 

# Engagement metrics
- like_count
- quoted_count
- retweet_count
- view_count

In [10]:
musk_twitter_data_nlp['date'] = pd.to_datetime(musk_twitter_data_nlp['date'])

# Gruppieren nach Tagesdatum und Summierung der gewünschten Spalten
#TODO Malte: viewCount wieder aus dem final_daily_df rausschmeißen
#TODO Malte: replyCount rein
engagement_metrics = (
    musk_twitter_data_nlp
    .groupby(musk_twitter_data_nlp['date'].dt.date)[['likeCount', 'quoteCount', 'retweetCount', 'viewCount']]
    .sum()
    .reset_index()
)

engagement_metrics.head()

,date,likeCount,quoteCount,retweetCount,viewCount
0,2015-01-05,3575.0,3.0,3625.0,0.0
1,2015-01-06,2268.0,6.0,1958.0,0.0
2,2015-01-10,21923.0,6.0,15895.0,0.0
3,2015-01-11,1342.0,2.0,1319.0,0.0
4,2015-01-12,4846.0,2.0,2967.0,0.0


# Tweet activity
New features:
- Number of tweets per day

In [5]:
tweet_counts_daily = (
    musk_twitter_data_all
    .groupby("date")
    .size()
    .reset_index(name="tweet_count")
)

# Sentiment Analysis

New Features:
- Poitve, Neutral ans Negative percentage of posts
- Polarization: Tweets with pos/neg > 0,6

In [ ]:
# Tweet sentiment analysis
# Model: https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment
model_name = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

def preprocess(text):
    return text.replace("\n", " ").strip()

def get_sentiment_probs(text):
    text = preprocess(text)
    tokens = tokenizer(text, return_tensors='pt', truncation=True)
    with torch.no_grad():
        output = model(**tokens)
    probs = softmax(output.logits.cpu().numpy()[0])
    return {
        "sentiment": ['negative', 'neutral', 'positive'][probs.argmax()],
        "neg": probs[0],
        "neu": probs[1],
        "pos": probs[2],
    }

def polarized_label(row):
    return "polarized" if max(row["pos"], row["neg"]) > 0.6 else "not_polarized"

results = musk_twitter_data_nlp["text_raw"].progress_apply(get_sentiment_probs).apply(pd.Series)
musk_twitter_data_nlp = pd.concat([musk_twitter_data_nlp, results], axis=1)
musk_twitter_data_nlp["sentiment_polarity"] = musk_twitter_data_nlp.apply(polarized_label, axis=1)

# 1) Unweighted daily aggregation (mean)
sentiment_avg = (musk_twitter_data_nlp.groupby("date")[["neg", "neu", "pos"]].mean().reset_index())
nlp_counts = (musk_twitter_data_nlp.groupby("date").size().reset_index(name="nlp_tweet_count"))
polar_mean = (musk_twitter_data_nlp.groupby("date")["sentiment_polarity"].value_counts(normalize=True).unstack(fill_value=0).reset_index())

sentiment_daily = (
    sentiment_avg
    .merge(nlp_counts, on="date", how="left")
    .merge(polar_mean, on="date", how="left")
)

# 2a) Weighted daily aggregation
weighted_sums = (
    musk_twitter_data_nlp
    .assign(
        neg_w = lambda df: df["neg"] * df["engagement_index"],
        neu_w = lambda df: df["neu"] * df["engagement_index"],
        pos_w = lambda df: df["pos"] * df["engagement_index"],
    )
    .groupby("date")
    .agg(
        neg_w_sum        = ("neg_w", "sum"),
        neu_w_sum        = ("neu_w", "sum"),
        pos_w_sum        = ("pos_w", "sum"),
        total_engagement = ("engagement_index", "sum"),
    )
    .reset_index()
    .assign(
        neg_weighted = lambda df: df["neg_w_sum"] / df["total_engagement"],
        neu_weighted = lambda df: df["neu_w_sum"] / df["total_engagement"],
        pos_weighted = lambda df: df["pos_w_sum"] / df["total_engagement"],
    )
)

# 2b) Weighted Polarisation pro Tag
polar_weighted = (
    musk_twitter_data_nlp
    .groupby(["date", "sentiment_polarity"])["engagement_index"]
    .sum()
    .reset_index(name="eng_w_sum")
    .pivot(index="date", columns="sentiment_polarity", values="eng_w_sum")
    .fillna(0)
    .rename(columns={
        "polarized": "polarized_eng",
        "not_polarized": "not_polarized_eng"
    })
    .reset_index()
    .merge(
        weighted_sums[["date", "total_engagement"]],
        on="date",
        how="left"
    )
    .assign(
        polarized_weighted     = lambda df: df["polarized_eng"]     / df["total_engagement"],
        not_polarized_weighted = lambda df: df["not_polarized_eng"] / df["total_engagement"]
    )
    [["date", "polarized_weighted", "not_polarized_weighted"]]
)

# 2c) Finales gewichtetes Daily-DF
sentiment_daily_weighted = (
    weighted_sums[[
        "date",
        "neg_weighted",
        "neu_weighted",
        "pos_weighted"
    ]]
    .merge(polar_weighted, on="date", how="left")
    .merge(nlp_counts,        on="date", how="left")
)

  0%|          | 0/4122 [00:00<?, ?it/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


KeyError: 'engagement_index'

# Emotions & Personality

New Features:
- Ekman Emotions: anger, disgust, fear, joy, neutral, sadness, surprise
- Big 5 personality traits: Extroversion, Neuroticism, Agreeableness, Conscientiousness, Openness

In [ ]:
# Ekman Emotionen
# Model: https://huggingface.co/j-hartmann/emotion-english-distilroberta-base
model_name = "j-hartmann/emotion-english-distilroberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

emotion_labels = ['anger', 'disgust', 'fear', 'joy', 'neutral', 'sadness', 'surprise']

# Helper that returns a dict of probabilities
def get_emotions(text):
    tokens = tokenizer(text, return_tensors='pt', truncation=True)
    with torch.no_grad():
        logits = model(**tokens).logits
    probs = softmax(logits.numpy()[0])
    return dict(zip(emotion_labels, probs))

# Apply to every tweet
print("Calculating emotion probabilities...")
emotion_probs = musk_twitter_data_nlp['text_raw'].progress_apply(get_emotions).apply(pd.Series)

# Append those new columns back onto your original DF
musk_twitter_data_nlp = pd.concat([musk_twitter_data_nlp.reset_index(drop=True), emotion_probs],axis=1)

# Unweighted daily aggregation (mean)
print("Aggregating daily emotions...")
emotion_daily = (musk_twitter_data_nlp.groupby('date')[emotion_labels].mean().reset_index())

# Weighted daily aggregation
weighted_sums = (
    musk_twitter_data_nlp
    .assign(**{
        f"{emo}_w": musk_twitter_data_nlp[emo] * musk_twitter_data_nlp["engagement_index"]
        for emo in emotion_labels
    })
    .groupby("date")
    .agg(
        **{f"{emo}_w_sum": (f"{emo}_w", "sum") for emo in emotion_labels},
        total_engagement = ("engagement_index", "sum"),
    )
    .reset_index()
)

emotion_daily_weighted = (
    weighted_sums
    .assign(**{
        f"{emo}_weighted": lambda df, emo=emo: df[f"{emo}_w_sum"] / df["total_engagement"]
        for emo in emotion_labels
    })
    [[
        "date",
        *[f"{emo}_weighted" for emo in emotion_labels]
    ]]
)

Calculating emotion probabilities...


  0%|          | 0/41847 [00:00<?, ?it/s]

Aggregating daily emotions...


In [ ]:
# Big Five Personality Traits
# Model: https://huggingface.co/Minej/bert-base-personality
tokenizer = BertTokenizer.from_pretrained("Minej/bert-base-personality")
model = BertForSequenceClassification.from_pretrained("Minej/bert-base-personality")

personality_labels = ['Extroversion', 'Neuroticism', 'Agreeableness', 'Conscientiousness', 'Openness']

def get_personality(text):
    inputs = tokenizer(text, truncation=True, padding=True, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    probs = sigmoid(outputs.logits).squeeze().numpy()
    return dict(zip(personality_labels, probs))

# Apply to every tweet
print("Calculating personality traits...")
personality_probs = musk_twitter_data_nlp['text_raw'].progress_apply(get_personality).apply(pd.Series)

# Append those new columns back onto your original DF
musk_twitter_data_nlp = pd.concat([musk_twitter_data_nlp.reset_index(drop=True), personality_probs], axis=1)

# Unweighted daily aggregation (mean)
print("Aggregating daily personality...")
personality_daily = (musk_twitter_data_nlp.groupby('date')[personality_labels].mean().reset_index())

# Weighted daily aggregation
weighted_sums = (
    musk_twitter_data_nlp
    .assign(**{
        f"{pers}_w": musk_twitter_data_nlp[pers] * musk_twitter_data_nlp["engagement_index"]
        for pers in personality_labels
    })
    .groupby("date")
    .agg(
        **{f"{pers}_w_sum": (f"{pers}_w", "sum") for pers in personality_labels},
        total_engagement = ("engagement_index", "sum"),
    )
    .reset_index()
)

personality_daily_weighted = (
    weighted_sums
    .assign(**{
        f"{pers}_weighted": lambda df, pers=pers: df[f"{pers}_w_sum"] / df["total_engagement"]
        for pers in personality_labels
    })
    [[
        "date",
        *[f"{pers}_weighted" for pers in personality_labels]
    ]]
)

Calculating personality traits...


  0%|          | 0/41847 [00:00<?, ?it/s]

Aggregating daily personality...


# Topic and word counts

New Features: 
- Daily Word counts
    - Rationale of Definition of words:
        - Company/ticker terms (e.g. tesla, tsla, spacex) capture direct references to publicly traded entities.
        - Product names (e.g. model, cybertruck, starship) often precede news that can move stock prices.
        - Crypto tokens (e.g. bitcoin, dogecoin, ethereum, crypto) map to Musk-driven volatility in the digital-asset markets
        - Financial keywords (e.g. stock, market, price, profit, loss, revenue) directly signal earnings or valuation discussions.
        - Macro terms (e.g. inflation, interest) reflect broader economic commentary that can sway sentiment.
        - Action verbs (buy, sell) often presage trading intent or recommendations.
- Topics of posts

In [ ]:
# Words
def tokenize(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|@\S+|[^a-z\s]", "", text)
    return text.split()

all_tokens = musk_twitter_data_nlp["text_lemmatized"].dropna().apply(tokenize)
flat_tokens = [token for sublist in all_tokens for token in sublist]
word_counts = Counter(flat_tokens)
word_counts = (
    pd.DataFrame(word_counts.items(), columns=["word", "count"])
      .sort_values("count", ascending=False)
      .reset_index(drop=True)
)

top20 = [
    'tesla', 'stock', 'market', 'price', 'profit', 'loss', 'revenue',
    'inflation', 'interest', 'bitcoin', 'dogecoin', 'crypto', 'ethereum',
    'spacex', 'model', 'cybertruck', 'starship', 'buy', 'sell'
]

top_word_df = musk_twitter_data_nlp.dropna(subset=['text_lemmatized']).copy()
top_word_df['tokens'] = top_word_df['text_lemmatized'].apply(tokenize)
top_word_df = top_word_df.explode('tokens')
top_word_df['tokens'] = top_word_df['tokens'].replace({'tsla': 'tesla'})

top_word_df = top_word_df[top_word_df['tokens'].isin(top20)].copy()

daily_word_counts = (
    top_word_df
    .groupby(['date','tokens'])
    .size()
    .unstack(fill_value=0)
)

daily_word_counts = daily_word_counts.reindex(
    columns=top20,
    fill_value=0
).sort_index()

In [ ]:
# Topic Analysis
# Model: https://huggingface.co/cardiffnlp/tweet-topic-21-multi
model_name = "cardiffnlp/tweet-topic-21-multi"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

topic_labels = [
    "arts_culture", "business_entrepreneurs", "celebrity_pop_culture",
    "diaries_daily_life", "family", "fashion_style", "film_tv_video",
    "fitness_&_health", "food_&_dining", "gaming", "learning_educational",
    "music", "news_social_concern", "other_hobbies", "relationships",
    "science_technology", "sports", "travel_adventure", "youth_student_life"
]

def get_topics(text):
    tokens = tokenizer(text, truncation=True, padding=True, return_tensors="pt")
    with torch.no_grad():
        output = model(**tokens)
    probs = softmax(output.logits.numpy()[0])
    return dict(zip(topic_labels, probs))

# Apply to every tweet
print("Calculating topic probabilities...")
topic_scores = musk_twitter_data_nlp['text_lemmatized'].progress_apply(get_topics).apply(pd.Series)

# Append those new columns back onto your original DF
musk_twitter_data_nlp = pd.concat([musk_twitter_data_nlp.reset_index(drop=True), topic_scores], axis=1)

# Unweighted daily aggregation (mean)
print("Aggregating daily topics...")
topics_daily = (musk_twitter_data_nlp.groupby('date')[topic_labels].mean().reset_index())

# Weighted daily aggregation
weighted_sums = (
    musk_twitter_data_nlp
    .assign(**{
        f"{top}_w": musk_twitter_data_nlp[top] * musk_twitter_data_nlp["engagement_index"]
        for top in topic_labels
    })
    .groupby("date")
    .agg(
        **{f"{top}_w_sum": (f"{top}_w", "sum") for top in topic_labels},
        total_engagement = ("engagement_index", "sum"),
    )
    .reset_index()
)

topics_daily_weighted = (
    weighted_sums
    .assign(**{
        f"{top}_weighted": lambda df, top=top: df[f"{top}_w_sum"] / df["total_engagement"]
        for top in topic_labels
    })
    [[
        "date",
        *[f"{top}_weighted" for top in topic_labels]
    ]]
)

Calculating topic probabilities...


  0%|          | 0/41847 [00:00<?, ?it/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Aggregating daily topics...


## Additional Features to consider/ ToDos
-  Done Tonalität / Sentiment (positiv, negativ, neutral) 
    - Ggfs. Auch Musks Stimmungswandel im Zeitverlauf messbar 
- OoS: Bullishness-Index (logarithmisches Verhältnis positiv/negativ)
    - Ist eher um den gesammten Markt zu analysieren
    - Das brauchen wir hier bei Elon Musk eigentlich nicht 
    - Vielleicht eher einsetzbar, wenn es über einen Tag gemessen wird 
- ToDo Tweet-Typ (z. B. Meme, Information, Ankündigung, Meinung, Engagement)
    - Studien zeigen, dass z. B. Meme-Posts und ironische Tweets besonders starke Kursreaktionen auslösen 
    - Bei Musk besonders relevant, da sein Kommunikationsstil sich im Zeitverlauf stark verändert hat 
- Done Zielrichtung des Tweets (z. B. Bitcoin, Dogecoin, Tesla, allgemeine Märkte) 
- Done: Zeitpunkt und Frequenz 
    - Wenn wir uns nur jeden Tag anschauen, dann kann dir Uhrzeit des Tweets nicht berücksichtigt werden oder? 
- OoS: Engagement-Formate (z. B. Umfragen, Fragen, Call-to-Actions) Fake News / Falschaussagen 
    - Wird schwierig zu beweisen sein 
- ToDo: like count

Aus Termin mit Peter:
- Einflussreiche weitere Personen: Kann man ggf auch aus quotes nehmen, ist mir nicht mehr ganz klar was er wollte.
- Quotes mit einbeziehen, Quote dataset enthält die texte der Quotes -> Einbeziehen, höhere Genauigkeit bei eg toics
- Normalisierung sollte hier mit rein
- Visualisierung ist noch nicht fertig

# neue Features zur bestehenden CSV hinzufügen

In [21]:
if einzelne_features_zur_bestehenden_CSV_hinzufügen:
    # 1. Final-Dataset laden mit geparster Datumsspalte
    final_daily_df = pd.read_csv("Data/twitter_data/processed/final_daily_df.csv", parse_dates=["date"])

    # 2. Platzhaltervariable für zusätzliche Feature-DataFrames
    # Beispiel: zusatz_feature_dfs = [df_neues_feature_1, df_neues_feature_2, ...]
    zusatz_feature_dfs = [
        
        # HIER DIE OBEN ERSTELLTEN NEUEN SPALTEN (inkl. 'date' spalte) AUFLISTEN
        engagement_metrics
    ]

    # 3. Iterativ mergen
    for feature_df in zusatz_feature_dfs:
        
        feature_df["date"] = pd.to_datetime(feature_df["date"])
        
        # Prüfen auf doppelte Spalten (außer 'date')
        doppelte = [col for col in feature_df.columns if col != "date" and col in final_daily_df.columns]
        if doppelte:
            raise ValueError(f"Die folgenden Spalten sind bereits in final_daily_df vorhanden und sollten evtl. nicht erneut gemerged werden: {doppelte}")

        # Merge auf 'date'
        final_daily_df = pd.merge(final_daily_df, feature_df, on="date", how="left")

    # 4. Ergebnis zurückschreiben
    final_daily_df.to_csv("Data/twitter_data/processed/final_daily_df.csv", index=False)


ValueError: Die folgenden Spalten sind bereits in final_daily_df vorhanden und sollten evtl. nicht erneut gemerged werden: ['likeCount', 'quoteCount', 'retweetCount', 'viewCount']

# Merge and Create final df
Merge the daily dfs in one dataframe and create csv

In [ ]:
# Merge with complete date, fill missing days with zero
if vollstaendige_neuerstellung_der_csv:
    # Unweighted final daily DataFrame
    final_daily_df = final_daily_df_base.merge(tweet_counts_daily, on="date", how="left").fillna(0)
    final_daily_df = final_daily_df.merge(engagement_metrics, on="date", how="left")
    final_daily_df["tweet_count"] = final_daily_df["tweet_count"].astype(int)
    final_daily_df = final_daily_df.merge(sentiment_daily, on="date", how="left")
    final_daily_df = final_daily_df.merge(emotion_daily, on="date", how="left")
    final_daily_df = final_daily_df.merge(personality_daily, on="date", how="left")
    final_daily_df = final_daily_df.merge(daily_word_counts, on="date", how="left")
    final_daily_df = final_daily_df.merge(topics_daily, on="date", how="left")
    final_daily_df["no_tweets"] = (final_daily_df["tweet_count"] == 0).astype(int)
    display(final_daily_df.info())
    display(final_daily_df.head())
    # Weighted final daily DataFrame
    weighted_final_daily_df = final_daily_df_base.merge(tweet_counts_daily, on="date", how="left").fillna(0)
    weighted_final_daily_df = weighted_final_daily_df.merge(engagement_metrics, on="date", how="left")
    weighted_final_daily_df["tweet_count"] = weighted_final_daily_df["tweet_count"].astype(int)
    weighted_final_daily_df = weighted_final_daily_df.merge(sentiment_daily_weighted, on="date", how="left")
    weighted_final_daily_df = weighted_final_daily_df.merge(emotion_daily_weighted, on="date", how="left")
    weighted_final_daily_df = weighted_final_daily_df.merge(personality_daily_weighted, on="date", how="left")
    weighted_final_daily_df = weighted_final_daily_df.merge(daily_word_counts, on="date", how="left")
    weighted_final_daily_df = weighted_final_daily_df.merge(topics_daily_weighted, on="date", how="left")
    weighted_final_daily_df["no_tweets"] = (weighted_final_daily_df["tweet_count"] == 0).astype(int)

    display(weighted_final_daily_df.info())
    display(weighted_final_daily_df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3756 entries, 0 to 3755
Data columns (total 59 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   date                      3756 non-null   object 
 1   tweet_count               3756 non-null   int64  
 2   neg                       3001 non-null   float32
 3   neu                       3001 non-null   float32
 4   pos                       3001 non-null   float32
 5   nlp_tweet_count           3001 non-null   float64
 6   not_polarized             3001 non-null   float64
 7   polarized                 3001 non-null   float64
 8   anger                     3001 non-null   float32
 9   disgust                   3001 non-null   float32
 10  fear                      3001 non-null   float32
 11  joy                       3001 non-null   float32
 12  neutral                   3001 non-null   float32
 13  sadness                   3001 non-null   float32
 14  surprise

None

,date,tweet_count,neg,neu,pos,nlp_tweet_count,not_polarized,polarized,anger,disgust,...,gaming,learning_&_educational,music,news_&_social_concern,other_hobbies,relationships,science_&_technology,sports,travel_&_adventure,youth_&_student_life
0,2015-01-01,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-01-02,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-01-03,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2015-01-04,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2015-01-05,2,0.020146,0.926847,0.053006,2.0,1.0,0.0,0.011031,0.004537,...,0.001542,0.003858,0.001551,0.79691,0.011698,0.001117,0.104953,0.002432,0.009344,0.001631


# Normalization

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Feature groups
count_features = [
    "tweet_count", "nlp_tweet_count",
    "tesla", "tsla", "stock", "market", "price", "profit", "loss",
    "revenue", "inflation", "interest", "bitcoin", "dogecoin",
    "crypto", "ethereum", "spacex", "model", "cybertruck",
    "starship", "buy", "sell"
]

score_features = [
    # Sentiment
    "neg", "neu", "pos", "not_polarized", "polarized",
    # Emotion (Ekman)
    "anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise",
    # Big Five
    "Extroversion", "Neuroticism", "Agreeableness",
    "Conscientiousness", "Openness",
    # Topics (alle Topic‐Spalten aus final_daily_df)
    *final_daily_df.columns[40:59].tolist()
]

binary_features = ["no_tweets"]

# Pipelines for each feature group
count_pipeline = Pipeline([
    ("imputer_zero", SimpleImputer(strategy="constant", fill_value=0)),
    ("minmax_scaler", MinMaxScaler()),
])

score_pipeline = Pipeline([
    ("imputer_zero", SimpleImputer(strategy="constant", fill_value=0)),
    ("std_scaler", StandardScaler()),
])

binary_pipeline = Pipeline([
    ("imputer_zero", SimpleImputer(strategy="constant", fill_value=0)),
])

# 4) ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("counts", count_pipeline, count_features),
        ("scores", score_pipeline, score_features),
        ("binary", binary_pipeline, binary_features),
    ],
    remainder="passthrough"
)

full_pipeline = Pipeline([
    ("preprocessing", preprocessor),
])

X_final = full_pipeline.fit_transform(final_daily_df)
print("final_daily_df vorbereitet, Shape:", X_final.shape)



In [ ]:
if vollstaendige_neuerstellung_der_csv:
    final_daily_df.to_csv(os.path.join('processed', 'final_daily_df.csv'), index=False)
    weighted_final_daily_df.to_csv(os.path.join('processed', 'weighted_final_daily_df.csv'), index=False)